# Preprocessing and Filtering

In [ ]:
with open("luffy_dataset.txt", "r") as f:
    raw = f.read()

print(len(raw))

In [ ]:
print(raw[:600])

In [ ]:
blocks_raw = [b.strip() for b in raw.split("\n\n") if b.strip()]
print(len(blocks_raw))
print()
print(blocks_raw[5])

In [ ]:
from collections import Counter

speakers = []
for block in blocks_raw:
    lines = block.splitlines()
    if lines and lines[0].endswith(":"):
        speakers.append(lines[0][:-1])

speaker_counts = Counter(speakers)
for name, count in speaker_counts.most_common(20):
    print(f"  {name:<20} {count:>5}")

In [ ]:
import re

URL_RE = re.compile(r"https?://\S+|www\.\S+")
HTML_RE = re.compile(r"<[^>]+>")
WS_RE  = re.compile(r"\s+")

QUOTE_TRANSLATION = str.maketrans({
    "\u201c": '"',
    "\u201d": '"',
    "\u2018": "'",
    "\u2019": "'",
    "\u2014": "-",
    "\u2013": "-",
    "\u2026": "...",
    "\u00a0": " ",
})

def clean_line(line):
    line = line.translate(QUOTE_TRANSLATION)
    line = URL_RE.sub("", line)
    line = HTML_RE.sub("", line)
    line = WS_RE.sub(" ", line).strip()
    if len(line) >= 2 and (
        (line.startswith('"') and line.endswith('"'))
        or (line.startswith("'") and line.endswith("'"))
    ):
        line = line[1:-1].strip()
    line = re.sub(r"^\[[^\]]*\]\s*", "", line)
    line = re.sub(r"\s*\[[^\]]*\]$", "", line).strip()
    if not re.search(r"[A-Za-z]", line):
        return ""
    return line

In [ ]:
dirty_examples = [
    '\u201cI\'m gonna be King of the Pirates!\u201d',
    '[laughing] You can\'t be serious...',
    'Check https://onepiece.fandom.com for more info',
    'Hm\u2014what do you think?',
    '   too    many    spaces   ',
    '!!!',
]

for ex in dirty_examples:
    print(f"RAW    : {repr(ex)}")
    print(f"CLEANED: {repr(clean_line(ex))}")
    print()

In [ ]:
with open("dataset/processed/corpus_clean.txt", "r") as f:
    text = f.read()

blocks_clean = [b.strip() for b in text.split("\n\n") if b.strip()]
print(f"raw blocks     : {len(blocks_raw)}")
print(f"clean+deduped  : {len(blocks_clean)}")
print(f"corpus size    : {len(text):,} chars")

# Tokenization

In [ ]:
characters = sorted(list(set(text)))
vocab_size  = len(characters)
print(repr("".join(characters)))
print("vocab size:", vocab_size)

In [ ]:
char_to_idx = {ch: i for i, ch in enumerate(characters)}
idx_to_char = {i: ch for i, ch in enumerate(characters)}

encode = lambda xs: [char_to_idx[x] for x in xs]
decode = lambda xs: ''.join([idx_to_char[x] for x in xs])

In [ ]:
print(encode("hello world"))
print(decode(encode("hello world")))

In [ ]:
luffy_line = "Luffy:\nI'm gonna be King of the Pirates!"
print(encode(luffy_line))
print(decode(encode(luffy_line)))

# Tiktoken tokenizer used by openai

In [ ]:
import tiktoken

enc_gpt2   = tiktoken.get_encoding('gpt2')
enc_cl100k = tiktoken.get_encoding('cl100k_base')
enc_o200k  = tiktoken.get_encoding('o200k_base')

print("gpt2       ", enc_gpt2.n_vocab)
print("cl100k_base", enc_cl100k.n_vocab)
print("o200k_base ", enc_o200k.n_vocab)

In [ ]:
for name, enc in [("gpt2", enc_gpt2), ("cl100k_base", enc_cl100k), ("o200k_base", enc_o200k)]:
    toks = enc.encode("hello world")
    print(f"{name:<14}  {toks}  ->  {repr(enc.decode(toks))}")

In [ ]:
luffy = "I'm gonna be King of the Pirates!"
for name, enc in [("gpt2", enc_gpt2), ("cl100k_base", enc_cl100k), ("o200k_base", enc_o200k)]:
    toks = enc.encode(luffy)
    pieces = [enc.decode([t]) for t in toks]
    print(f"{name:<14}  {len(toks)} tokens: {pieces}")

In [ ]:
test_lines = [
    "I'm gonna be King of the Pirates!",
    "Gomu Gomu no Pistol!",
    "I don't care about that. I just want to be free.",
    "Zoro:\nNothing happened.",
]

for line in test_lines:
    print(repr(line))
    print(f"  char-level    {len(encode(line))}")
    for name, enc in [("gpt2", enc_gpt2), ("cl100k_base", enc_cl100k), ("o200k_base", enc_o200k)]:
        print(f"  {name:<14} {len(enc.encode(line))}")
    print()

In [ ]:
n_char = len(encode(text))
print(f"char-level    {n_char:>10,}  chars/tok={len(text)/n_char:.2f}  vocab={vocab_size}")
for name, enc in [("gpt2", enc_gpt2), ("cl100k_base", enc_cl100k), ("o200k_base", enc_o200k)]:
    n = len(enc.encode(text))
    print(f"{name:<14} {n:>10,}  chars/tok={len(text)/n:.2f}  vocab={enc.n_vocab:,}")

### Byte-pair encoding (BPE) is a simple form of data compression in which the most common pair of consecutive bytes of data is replaced with a byte that does not occur in that data. The replacement byte is then used in the data stream to represent that pair of bytes, and the original pair of bytes is restored by the inverse process. BPE is a form of dictionary coding, and is capable of replicating the entire data stream from the dictionary alone, with no other additional information.

In [ ]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [ ]:
bpe_vocab_size = 40
num_merges     = vocab_size - bpe_vocab_size
tokens = encode(text[:200_000])
ids    = list(tokens)

vocab  = idx_to_char.copy()
merges = {}

for i in range(num_merges):
    stats      = get_stats(ids)
    pair       = max(stats, key=stats.get)
    idx        = vocab_size + i
    vocab[idx] = vocab[pair[0]] + vocab[pair[1]]
    print(f"merging {pair} into a new token {idx}")
    ids        = merge(ids, pair, idx)
    merges[pair] = idx

In [ ]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

In [ ]:
vocab

In [ ]:
def decode_bpe(ids):
    return "".join(vocab[idx] for idx in ids)

def encode_bpe(text):
    tokens = list(encode(text))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair  = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        tokens = merge(tokens, pair, merges[pair])
    return tokens

In [ ]:
encode_bpe("hi there")

In [ ]:
encode("hi there")

In [ ]:
decode_bpe(encode_bpe("I'm gonna be King of the Pirates!"))

In [ ]:
text_check = "A slightly-modified version of the algorithm is used in large language model tokenizers."
decode_bpe(encode_bpe(text_check)) == text_check